In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress

# Load Excel data (ensure file is in the same directory)
df = pd.read_excel('data/f1-race-data.xlsx')

# Strip extra whitespace from column headers
df.rename(columns=lambda x: x.strip(), inplace=True)

# Convert lap time to seconds if needed
def time_to_seconds(t):
    try:
        m, s = map(float, str(t).split(":"))
        return m * 60 + s
    except:
        return None

if 'LapTime_sec' not in df.columns:
    df['LapTime_sec'] = df['LapTime'].apply(time_to_seconds)


In [ ]:
print(df.columns.tolist())


In [ ]:
"Rainy, 22.0°C, 65% Humidity"


In [ ]:
df.rename(columns=lambda x: x.strip(), inplace=True)  # Remove leading/trailing spaces from all column names


In [ ]:

df['Weather_Type'] = df['Weather Condition'].str.extract(r'^(Rainy|Dry)')



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
avg_laptimes = df.groupby('Weather_Type')['LapTime_sec'].mean().reset_index()

plt.figure(figsize=(6,4))
sns.barplot(data=avg_laptimes, x='Weather_Type', y='LapTime_sec')
plt.title('Average Lap Time by Weather Type')
plt.ylabel('Lap Time (seconds)')
plt.show()



In [ ]:
from scipy.stats import linregress

# Function to compute the slope of LapTime vs LapNumber
def compute_degradation_slope(group):
    if group['LapNumber'].nunique() > 1:
        slope, _, _, _, _ = linregress(group['LapNumber'], group['LapTime_sec'])
        return slope
    return None

# Group by Compound and Weather and compute degradation
degradation = (
    df.dropna(subset=['LapNumber', 'LapTime_sec'])
    .groupby(['Compound', 'Weather_Type'])
    .apply(compute_degradation_slope)
    .reset_index(name='Degradation_Slope')
)

# Drop rows where slope couldn't be computed
degradation.dropna(inplace=True)

# Visualize
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
sns.barplot(data=degradation, x='Compound', y='Degradation_Slope', hue='Weather_Type')
plt.title('Tire Degradation Rate by Compound and Weather')
plt.ylabel('Slope of Lap Time vs Lap Number')
plt.xlabel('Tire Compound')
plt.grid(True)
plt.show()


In [ ]:
# Ensure no division by zero
df = df[df['Laps'] > 0]

# Create Aggressiveness Score
df['Aggressiveness_Score'] = df['Overtakes'] / df['Laps']

# Group by weather
agg_score = df.groupby('Weather_Type')['Aggressiveness_Score'].mean().reset_index()

# Plot
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,4))
sns.barplot(data=agg_score, x='Weather_Type', y='Aggressiveness_Score')
plt.title('Aggressiveness Score by Weather Type')
plt.ylabel('Avg Overtakes per Lap')
plt.grid(True)
plt.show()


In [ ]:
# Ensure no missing values
df = df.dropna(subset=['PitTime'])

# Calculate Pit Deviation (how much faster/slower than race avg)
df['PitDeviation'] = df.groupby('Grand_Prix')['PitTime'].transform('mean') - df['PitTime']

# Group by weather type
pit_eff = df.groupby('Weather_Type')['PitDeviation'].mean().reset_index()

# Plot
plt.figure(figsize=(6,4))
sns.barplot(data=pit_eff, x='Weather_Type', y='PitDeviation')
plt.title('Pit Stop Efficiency by Weather Type')
plt.ylabel('Average Deviation from Race Pit Time (sec)')
plt.grid(True)
plt.show()


In [ ]:
print(df.columns.tolist())


In [ ]:
# Average number of stops by weather
stops_by_weather = df.groupby('Weather_Type')['Stops'].mean().reset_index()

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6,4))
sns.barplot(data=stops_by_weather, x='Weather_Type', y='Stops')
plt.title('Average Pit Stops by Weather Type')
plt.ylabel('Average Stops')
plt.grid(True)
plt.show()


In [ ]:
# Calculate rolling average lap time (3-lap window) per driver and race
df['RollingAvgLap'] = df.groupby(['Driver', 'Grand_Prix'])['LapTime_sec'].transform(
    lambda x: x.rolling(window=3, min_periods=1).mean()
)


In [ ]:
# Flag laps where LapTime is 40% higher than recent average
df['Pit_Lap_Flag'] = df['LapTime_sec'] > 1.4 * df['RollingAvgLap']


In [ ]:
df['Pit_Lap_Flag'].value_counts()


In [ ]:
# Flag laps with LapTime > 95 sec as pit stops
df['Pit_Lap_Flag'] = df['LapTime_sec'] > 95


In [ ]:
df['Pit_Lap_Flag'].value_counts()


In [ ]:
pit_flag_summary = df.groupby('Weather_Type')['Pit_Lap_Flag'].mean().reset_index()
pit_flag_summary['Pit_Lap_Flag'] *= 100  # convert to percentage

import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick

plt.figure(figsize=(6,4))
sns.barplot(data=pit_flag_summary, x='Weather_Type', y='Pit_Lap_Flag')
plt.title('% of Laps Flagged as Pit Stops by Weather')
plt.ylabel('Percent of Pit-Like Laps')
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter())
plt.grid(True)
plt.show()


In [ ]:
# Fix column name spaces
df.columns = df.columns.str.strip()

# Recalculate lap time in seconds
df['LapTime_sec'] = pd.to_timedelta(df['LapTime'], errors='coerce').dt.total_seconds()

# Extract Rainy/Dry
df['Weather_Type'] = df['Weather Condition'].str.extract(r'^(Rainy|Dry)')

# Calculate Aggressiveness Score
df = df[df['Laps'] > 0]
df['Aggressiveness_Score'] = df['Overtakes'] / df['Laps']


In [ ]:
df['Temperature'] = df.iloc[:, 40].astype(str).str.extract(r'(\d+)').astype(float)


In [ ]:
# Extract temperature from "18°C"
df['Temperature'] = df.iloc[:, 40].astype(str).str.extract(r'(\d+)').astype(float)

# Normalize "Rain (Y/N)"
df['Rain'] = df.iloc[:, 41].astype(str).str.strip().str.lower().map({'yes': 1, 'no': 0})

# Convert Humidity (e.g., 0.75 → 75)
df['Humidity'] = pd.to_numeric(df.iloc[:, 42], errors='coerce')

# Create Weather Condition string
df['Weather Condition'] = df.apply(
    lambda row: f"{'Rainy' if row['Rain'] == 1 else 'Dry'}, {row['Temperature']}°C, {int(row['Humidity'] * 100)}% Humidity"
    if pd.notnull(row['Temperature']) and pd.notnull(row['Humidity']) else None,
    axis=1
)

# Extract "Rainy" or "Dry" for clustering
df['Weather_Type'] = df['Weather Condition'].str.extract(r'^(Rainy|Dry)')


In [ ]:
print(df.columns.tolist())


In [ ]:
df.columns = df.columns.str.strip()  # Strip any hidden spaces just in case


In [ ]:
df[['Driver', 'LapTime_sec', 'Compound', 'Weather_Type', 'Aggressiveness_Score']].dropna().shape


In [ ]:
cluster_df = df[['Driver', 'LapTime_sec', 'Compound', 'Weather_Type', 'Aggressiveness_Score']].dropna()

# Check if anything exists before continuing
if not cluster_df.empty:
    cluster_df = cluster_df.groupby(['Driver', 'Compound', 'Weather_Type'], as_index=False).mean()

    # Continue with clustering...
else:
    print("❌ No rows available for clustering — check missing data in LapTime or Aggressiveness.")


In [ ]:
df['Weather_Type'] = df.iloc[:, 41].astype(str).str.strip().str.lower().map({'yes': 'Rainy', 'no': 'Dry'})


In [ ]:
df[['LapTime_sec', 'Aggressiveness_Score']].isnull().sum()


In [ ]:
cluster_df = df[['Driver', 'LapTime_sec', 'Aggressiveness_Score']].dropna()
cluster_df = cluster_df.groupby('Driver', as_index=False).mean()


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans

pipeline = Pipeline([
    ('scale', StandardScaler()),
    ('kmeans', KMeans(n_clusters=3, random_state=42))
])

pipeline.fit(cluster_df[['LapTime_sec', 'Aggressiveness_Score']])
cluster_df['Cluster'] = pipeline.named_steps['kmeans'].labels_


In [ ]:
cluster_df.groupby('Cluster')[['LapTime_sec', 'Aggressiveness_Score']].mean().round(2)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.scatterplot(
    data=cluster_df,
    x='Aggressiveness_Score',
    y='LapTime_sec',
    hue='Cluster',
    palette='Set2',
    s=100
)

plt.title('Driver Clusters by Lap Time and Aggressiveness')
plt.xlabel('Aggressiveness Score')
plt.ylabel('Average Lap Time (seconds)')
plt.grid(True)
plt.legend(title='Cluster')
plt.show()


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Define your features and target
features = ['LapTime_sec', 'Aggressiveness_Score', 'Rain', 'Humidity', 'Temperature', 'Pit_Lap_Flag']
target = 'Position'

# Drop missing values
tree_df = df[features + [target]].dropna()

# Set up X and y
X = tree_df[features]
y = tree_df[target].astype(int)  # ensure target is integer for classification

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

# Confusion matrix
ConfusionMatrixDisplay.from_estimator(clf, X_test, y_test, cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

# Optional: print classification report
print(classification_report(y_test, clf.predict(X_test)))



In [ ]:
df[features + [target]].isna().sum()


In [ ]:
tree_df = df[features + [target]].copy()

# Fill missing weather-related features
tree_df['Rain'].fillna(0, inplace=True)  # assume dry
tree_df['Humidity'].fillna(tree_df['Humidity'].median(), inplace=True)
tree_df['Temperature'].fillna(tree_df['Temperature'].median(), inplace=True)

# Confirm no nulls
print(tree_df.isna().sum())


In [ ]:
X = tree_df[features]
y = tree_df[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

ConfusionMatrixDisplay.from_estimator(clf, X_test, y_test, cmap='Blues')
plt.title("Confusion Matrix")
plt.show()

print(classification_report(y_test, clf.predict(X_test)))


In [ ]:
tree_df = df[features + [target]].dropna()
print(tree_df.shape)


In [ ]:
tree_df = df[features + [target]].dropna()
print(tree_df.shape)


In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(40, 25))  # Increase size significantly
plot_tree(
    clf,
    feature_names=X.columns,
    class_names=[str(cls) for cls in clf.classes_],
    filled=True,
    rounded=True,
    fontsize=12  # Adjust font size for readability
)
plt.title("Decision Tree - Race Position Prediction", fontsize=20)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

fig = plt.figure(figsize=(40, 25))
plot_tree(
    clf,
    feature_names=X.columns,
    class_names=[str(cls) for cls in clf.classes_],
    filled=True,
    rounded=True,
    fontsize=9,
)

plt.title("Decision Tree - Race Position Prediction", fontsize=20)

# 🔽 Save as PDF
fig.savefig("decision_tree_race_position.pdf", format='pdf', bbox_inches='tight')

plt.show()


In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

fig = plt.figure(figsize=(40, 25))
plot_tree(
    clf,
    feature_names=X.columns,
    class_names=[str(cls) for cls in clf.classes_],
    filled=True,
    rounded=True,
    fontsize=9
)

plt.title("Decision Tree - Race Position Prediction", fontsize=20)

# 🔽 Save as PDF
fig.savefig("decision_tree_race_position.pdf", format='pdf', bbox_inches='tight')

plt.show()


In [ ]:
plot_tree(
    clf,
    feature_names=X.columns,
    class_names=[str(cls) for cls in clf.classes_],
    filled=True,
    rounded=True,
    max_depth=2,   # Only show top 3 levels
    fontsize=10,
)


In [ ]:
import pandas as pd

# Get feature importances
importances = clf.feature_importances_
features = X.columns

# Create a bar plot
plt.figure(figsize=(8, 5))
sns.barplot(x=importances, y=features)
plt.title("Feature Importances in Decision Tree")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.grid(True)
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Generate confusion matrix
cm = confusion_matrix(y_test, clf.predict(X_test))

# Optional: define labels if your 'Position' values are known (e.g., [1, 2, 3, ..., 13])
labels = sorted(y.unique())

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Decision Tree Confusion Matrix Heatmap')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


In [ ]:
def group_position(pos):
    if pos <= 3:
        return 'Podium'
    elif pos <= 8:
        return 'Midfield'
    else:
        return 'Backmarkers'

df['Position_Tier'] = df['Position'].apply(group_position)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
clf.fit(X_train, y_train)


In [ ]:
X = df[features]  # same feature columns as before
y = df['Position_Tier']  # use new grouped target


In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)


In [ ]:
features = ['LapTime_sec', 'Aggressiveness_Score', 'Rain', 'Humidity', 'Temperature', 'Pit_Lap_Flag']


In [ ]:
print(df[features + ['Position_Tier']].dropna().shape)


In [ ]:
features = ['LapTime_sec', 'Aggressiveness_Score', 'Pit_Lap_Flag']



In [ ]:
# Step 1: Impute (if needed, although you already dropped NaNs)
imputer = SimpleImputer(strategy='mean')
X_imputed = imputer.fit_transform(df[features])

# Step 2: Encode the categorical target
le = LabelEncoder()
y_encoded = le.fit_transform(df['Position_Tier'])

# Step 3: Apply SMOTE for class balancing
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_imputed, y_encoded)



In [ ]:
clf_rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
clf_rf.fit(X_resampled, y_resampled)


In [ ]:
# 1. Define features and target
features = ['LapTime_sec', 'Aggressiveness_Score', 'Rain', 'Humidity', 'Temperature', 'Pit_Lap_Flag']
target = 'Position_Tier'

# 2. Drop rows with missing target
df_clean = df[features + [target]].dropna(subset=[target])

# 3. Impute missing values in features
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean')
X = imputer.fit_transform(df_clean[features])

# 4. Encode target labels
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(df_clean[target])

# 5. Train-test split BEFORE applying SMOTE
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 6. Apply SMOTE only on training data
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# 7. Train the model
from sklearn.ensemble import RandomForestClassifier
clf_rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
clf_rf.fit(X_train_resampled, y_train_resampled)

# 8. Predict and evaluate
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

y_pred = clf_rf.predict(X_test)
print(classification_report(y_test, y_pred))
ConfusionMatrixDisplay.from_estimator(clf_rf, X_test, y_test, cmap='Blues')


In [ ]:
features = ['LapTime_sec', 'Aggressiveness_Score', 'Rain', 'Humidity', 'Temperature', 'Pit_Lap_Flag']
target = 'Position_Tier'

# Drop rows with NaNs
tree_df = df[features + [target]].dropna()

# Define X and y
X = tree_df[features]
y = tree_df[target]


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_enc = LabelEncoder()
y_encoded = label_enc.fit_transform(y)  # e.g., Tier 1 → 0, Tier 2 → 1, etc.


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.3, random_state=42)


In [ ]:
from sklearn.tree import DecisionTreeClassifier

clf = DecisionTreeClassifier(max_depth=4, random_state=42)
clf.fit(X_train, y_train)


In [ ]:
from sklearn.metrics import classification_report

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=label_enc.classes_))


In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(24, 12))  # Increase for larger tree
plot_tree(clf, 
          feature_names=features, 
          class_names=label_enc.classes_, 
          filled=True, 
          rounded=True,
          fontsize=9)
plt.title("Decision Tree – Predicting Driver Performance Tier")
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Get feature importances
importances = clf.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Bar plot
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df, palette='viridis')
plt.title('Feature Importance from Decision Tree')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix
import numpy as np

# Generate confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Plot heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_enc.classes_,
            yticklabels=label_enc.classes_)
plt.title('Confusion Matrix Heatmap')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()
